<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система строк заказов</h2>

----

### Вариант задания 15


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Крутые линии заказов для какой-нибудь таблицы.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public delegate void OrderLineEventHandler(OrderLine source, string message);
public delegate decimal PriceModifier(decimal basePrice, decimal factor);

public interface ITrackable
{
    string TrackingNumber { get; set; }
    void UpdateTracking(string status);
    void UpdateTracking(string status, string location);
}

public interface IReturnable
{
    int ReturnWindowDays { get; set; }
    bool RequestReturn(string reason);
}

public interface IOrderLine
{
    string ProductName { get; set; }
    decimal CalculateTotal();
    decimal GetShippingCost();
    void ApplyQualityCheck();
}

public class PackageSize
{
    public decimal Size { get; private set; }
    public decimal X1 { get; set; }
    public decimal Y1 { get; set; }
    public decimal Z1 { get; set; }
    public decimal X2 { get; set; }
    public decimal Y2 { get; set; }
    public decimal Z2 { get; set; }

    public PackageSize(decimal x1, decimal y1, decimal z1,
        decimal x2, decimal y2, decimal z2)
    {
        X1 = x1;
        Y1 = y1;
        Z1 = z1;
        X2 = x2;
        Y2 = y2;
        Z2 = z2;
        Size = Math.Abs((x2 - x1) * (y2 - y1) * (z2 - z1));
    }
}

public class OrderLine : IOrderLine
{
    public Guid ProductId { get; private set; } = Guid.NewGuid();
    public string ProductName { get; set; }
    public decimal Price { get; protected set; }
    public PackageSize Box { get; private set; } = new PackageSize(1, 1, 1, 2, 2, 2);
    public decimal Weight { get; set; }
    public bool IsFragile { get; set; }
    public string Category { get; set; }
    public string OriginCountry { get; set; }
    public DateTime CreatedDate { get; set; }
    public int PriorityLevel { get; set; }
    public bool IsExpressDelivery { get; set; }

    public string Barcode { get; set; }
    public string StorageConditions { get; set; }
    public bool IsInspected { get; set; }
    public decimal InsuranceValue { get; set; }

    public string CurrentStatus { get; set; }
    public HashSet<string> Tags { get; private set; } = new HashSet<string>();
    public List<string> InternalNotes { get; private set; } = new List<string>();
    public decimal DiscountPercent { get; set; }

    public event OrderLineEventHandler LineStatusChanged;

    public OrderLine(string productName, decimal price)
    {
        ProductName = productName;
        Price = price;
        Weight = 1.0M;
        IsFragile = false;
        Category = "Общая";
        OriginCountry = "Не указана";
        CreatedDate = DateTime.Now;
        PriorityLevel = 0;
        IsExpressDelivery = false;
        Barcode = "BAR-000000";
        StorageConditions = "Стандартные складские условия";
        IsInspected = false;
        InsuranceValue = price * 1.1M;
        CurrentStatus = "Создан";
        DiscountPercent = 0M;
    }

    public virtual decimal CalculateTotal()
    {
        return Price * 1.25M * 1.01M + 10;
    }

    public virtual decimal CalculateTotal(decimal discountPercent)
    {
        return (Price * 1.25M * 1.01M + 10) * (1M - discountPercent / 100M);
    }

    public virtual void UpdatePrice(decimal newPrice)
    {
        Price = newPrice;
    }

    public void UpdatePrice(decimal newPrice, decimal taxMultiplier)
    {
        Price = newPrice * taxMultiplier;
    }

    public void ChangeSizeBox(PackageSize box)
    {
        Box = box;
    }

    public virtual decimal GetShippingCost()
    {
        return IsExpressDelivery ? Weight * 10.0M + 25.0M : Weight * 5.0M;
    }

    public virtual decimal GetShippingCost(bool isExpress)
    {
        return isExpress ? Weight * 10.0M + 25.0M : Weight * 5.0M;
    }

    public virtual void ApplyQualityCheck()
    {
        Console.WriteLine($"Базовая проверка качества для товара: {ProductName}. Проверка пройдена.");
    }

    public void PrintWarning()
    {
        if (IsFragile)
        {
            Console.WriteLine($"Внимание: Товар {ProductName} хрупкий!");
        }
    }

    public string GetProductDetails()
    {
        return $"ID продукта: {ProductId}\n" +
            $"Название продукта: {ProductName}\n" +
            $"Себестоимость продукта: {Price}\n" +
            $"Размер упаковки для товара: {Box.Size} м^3\n" +
            $"Вес: {Weight} кг, Категория: {Category}\n" +
            $"Страна происхождения: {OriginCountry}, Приоритет: {PriorityLevel}, Статус: {CurrentStatus}\n" +
            $"Теги: {string.Join(", ", Tags)}\n";
    }

    public decimal CalculateInsurance()
    {
        return InsuranceValue * 0.02M;
    }

    public void MarkAsInspected()
    {
        IsInspected = true;
        Console.WriteLine($"Товар {ProductName} отмечен как прошедший первичный осмотр.");
    }

    public void UpdateStorageConditions(string conditions)
    {
        StorageConditions = conditions;
    }

    public void UpdateStatus(string newStatus)
    {
        CurrentStatus = newStatus;
        LineStatusChanged?.Invoke(this, $"Статус позиции '{ProductName}' изменен на '{newStatus}'.");
    }

    public void AddTag(string tag)
    {
        Tags.Add(tag);
    }

    public void AddInternalNote(string note)
    {
        InternalNotes.Add($"[{DateTime.Now:HH:mm:ss}] {note}");
    }

    public void ApplyCustomDiscount(decimal factor, PriceModifier modifier)
    {
        Price = modifier(Price, factor);
    }

    void IOrderLine.ApplyQualityCheck()
    {
        Console.WriteLine($"[Явная реализация IOrderLine] Проверка качества позиции: {ProductName}.");
        ApplyQualityCheck();
    }
}

public class StandardLine : OrderLine
{
    public int Units { get; set; }
    public string Supplier { get; set; }
    public DateTime RestockDate { get; set; }
    public int WarehouseZone { get; set; }
    public string QualityStatus { get; set; }

    public string BatchNumber { get; set; }
    public int MinStockThreshold { get; set; }
    public bool IsPalletized { get; set; }
    public decimal PalletWeight { get; set; }

    public List<string> SerialNumbers { get; private set; } = new List<string>();
    public int DefectCount { get; set; }
    public HashSet<string> InspectionCertificates { get; private set; } = new HashSet<string>();
    public int ShelfLifeDays { get; set; }

    public StandardLine(string productName, decimal price, int units)
        : base(productName, price)
    {
        Units = units;
        Supplier = "Неизвестен";
        RestockDate = DateTime.Now.AddDays(7);
        WarehouseZone = 1;
        QualityStatus = "Ожидает проверки";
        BatchNumber = "BATCH-2026-01";
        MinStockThreshold = 5;
        IsPalletized = false;
        PalletWeight = 15.0M;
        DefectCount = 0;
        ShelfLifeDays = 365;
    }

    public override decimal CalculateTotal()
    {
        return (Price * 1.25M * 1.01M + 10) * Units;
    }

    public override decimal CalculateTotal(decimal discountPercent)
    {
        return (Price * 1.25M * 1.01M + 10) * Units * (1M - discountPercent / 100M);
    }

    public override decimal GetShippingCost()
    {
        return IsExpressDelivery ? Weight * 8.0M * Units + 30.0M : base.GetShippingCost();
    }

    public override decimal GetShippingCost(bool isExpress)
    {
        return isExpress ? Weight * 8.0M * Units + 30.0M : base.GetShippingCost();
    }

    public override void ApplyQualityCheck()
    {
        QualityStatus = Units > 0 ? "Проверено на складе" : "Нет в наличии";
        Console.WriteLine($"Стандартная проверка партии товара {ProductName}: статус '{QualityStatus}'.");
    }

    public bool CheckAvailability()
    {
        return Units > 0;
    }

    public void ChangeUnits(int delta)
    {
        Units += delta;
        if (Units < 0) Units = 0;
    }

    public bool NeedsRestock()
    {
        return Units <= MinStockThreshold;
    }

    public void Palletize()
    {
        IsPalletized = true;
        Console.WriteLine($"Партия {ProductName} (партия: {BatchNumber}) упакована на паллету.");
    }

    public decimal GetTotalWeightWithPallet()
    {
        return Weight * Units + (IsPalletized ? PalletWeight : 0M);
    }

    public void RegisterSerialNumber(string serial)
    {
        SerialNumbers.Add(serial);
    }

    public void RecordDefect(int count = 1)
    {
        DefectCount += count;
        ChangeUnits(-count);
        Console.WriteLine($"Зафиксирован брак для {ProductName}: {count} шт. Доступно: {Units} шт.");
    }

    public void AddCertificate(string certNumber)
    {
        InspectionCertificates.Add(certNumber);
    }

    public bool ValidateCertificates(int minimumRequired)
    {
        return InspectionCertificates.Count >= minimumRequired;
    }
}

public class BulkyStandardLine : StandardLine, ITrackable, IReturnable
{
    public decimal HandlingFee { get; set; }
    public string TrackingNumber { get; set; }
    public int ReturnWindowDays { get; set; }
    public int RequiredLoaders { get; set; }
    public string TransportType { get; set; }

    public decimal MaxFloorLoad { get; set; }
    public bool RequiresCrane { get; set; }
    public string RoutePlan { get; set; }
    public decimal OversizeRate { get; set; }

    public List<string> Checkpoints { get; private set; } = new List<string>();
    public bool AssemblyInstructionsRequired { get; set; }
    public string UnloadingPermitNumber { get; set; }
    public HashSet<string> SpecialEquipmentList { get; private set; } = new HashSet<string>();

    public BulkyStandardLine(string productName, decimal price, int units, decimal handlingFee)
        : base(productName, price, units)
    {
        HandlingFee = handlingFee;
        TrackingNumber = "-";
        ReturnWindowDays = 14;
        RequiredLoaders = 2;
        TransportType = "Стандартный грузовик";
        MaxFloorLoad = 1500M;
        RequiresCrane = false;
        RoutePlan = "Прямой магистральный маршрут";
        OversizeRate = 1.15M;
        AssemblyInstructionsRequired = false;
        UnloadingPermitNumber = "БЕЗ_РАЗРЕШЕНИЯ";
    }

    public override decimal CalculateTotal()
    {
        return base.CalculateTotal() + HandlingFee;
    }

    public override decimal CalculateTotal(decimal discountPercent)
    {
        return base.CalculateTotal(discountPercent) + HandlingFee;
    }

    public override decimal GetShippingCost()
    {
        return base.GetShippingCost() + (IsExpressDelivery ? 100.0M : 50.0M);
    }

    public override decimal GetShippingCost(bool isExpress)
    {
        return isExpress ? base.GetShippingCost(true) + 100.0M : base.GetShippingCost() + 50.0M;
    }

    public override void ApplyQualityCheck()
    {
        Console.WriteLine($"Усиленная проверка крупногабаритного груза {ProductName} (грузчиков: {RequiredLoaders}, транспорт: {TransportType}).");
    }

    public void AssignCrane()
    {
        RequiresCrane = true;
        Console.WriteLine($"Для погрузки крупногабаритного товара {ProductName} назначен автокран.");
    }

    public decimal CalculateOversizeSurcharge()
    {
        return HandlingFee * (OversizeRate - 1M);
    }

    public void UpdateRoute(string newRoute)
    {
        RoutePlan = newRoute;
        Console.WriteLine($"Маршрут крупногабаритного груза изменен: {RoutePlan}");
    }

    public void AddCheckpoint(string point)
    {
        Checkpoints.Add(point);
    }

    public void AddEquipment(string equipment)
    {
        SpecialEquipmentList.Add(equipment);
    }

    public decimal EstimateRiggingCost()
    {
        return RequiredLoaders * 1200M + (RequiresCrane ? 5000M : 0M);
    }

    public void IssueUnloadingPermit(string permit)
    {
        UnloadingPermitNumber = permit;
        Console.WriteLine($"Выдано разрешение на разгрузку {ProductName}: {permit}");
    }

    void ITrackable.UpdateTracking(string status)
    {
        Console.WriteLine($"[Явная реализация ITrackable] Статус доставки ({TrackingNumber}): {status}");
    }

    void ITrackable.UpdateTracking(string status, string location)
    {
        Console.WriteLine($"[Явная реализация ITrackable] Статус доставки ({TrackingNumber}): {status}, местоположение: {location}");
    }

    bool IReturnable.RequestReturn(string reason)
    {
        Console.WriteLine($"[Явная реализация IReturnable] Запрос на возврат {ProductName} по причине: {reason}");
        return true;
    }
}

public class SpecialLine : OrderLine
{
    public int PercentDiscount { get; set; }
    public string PromoCode { get; set; }
    public DateTime ExpirationDate { get; set; }
    public decimal MinOrderAmount { get; set; }
    public string PromotionName { get; set; }

    public int LoyaltyPointsAwarded { get; set; }
    public string TargetAudience { get; set; }
    public bool IsCombinableWithOtherPromos { get; set; }
    public decimal MaxDiscountCap { get; set; }

    public HashSet<string> EligibleCustomerCategories { get; private set; } = new HashSet<string>();
    public List<string> PromoHistory { get; private set; } = new List<string>();
    public decimal CashbackPercent { get; set; }
    public int MaxUsageLimit { get; set; }

    public SpecialLine(string productName, decimal price, int percentDiscount)
        : base(productName, price)
    {
        PercentDiscount = percentDiscount;
        PromoCode = "ПРОМОКОД_ОТСУТСТВУЕТ";
        ExpirationDate = DateTime.Now.AddMonths(1);
        MinOrderAmount = 0M;
        PromotionName = "Обычная акция";
        LoyaltyPointsAwarded = 50;
        TargetAudience = "Постоянные клиенты";
        IsCombinableWithOtherPromos = false;
        MaxDiscountCap = 90M;
        CashbackPercent = 3.0M;
        MaxUsageLimit = 100;
    }

    public override decimal CalculateTotal()
    {
        return (Price * 1.25M * 1.01M + 10) * (1M - PercentDiscount / 100M);
    }

    public override decimal CalculateTotal(decimal additionalDiscount)
    {
        decimal totalDiscount = PercentDiscount + additionalDiscount;
        if (totalDiscount > MaxDiscountCap) totalDiscount = MaxDiscountCap;
        return (Price * 1.25M * 1.01M + 10) * (1M - totalDiscount / 100M);
    }

    public override void UpdatePrice(decimal newPrice)
    {
        Price = newPrice * (1M - PercentDiscount / 100M);
    }

    public void ApplyPromo(string code)
    {
        if (code == PromoCode && DateTime.Now <= ExpirationDate)
        {
            PercentDiscount += 5;
        }
    }

    public void ApplyPromo(string code, int extraDiscount)
    {
        if (code == PromoCode && DateTime.Now <= ExpirationDate)
        {
            PercentDiscount += extraDiscount;
        }
    }

    public override void ApplyQualityCheck()
    {
        Console.WriteLine($"Проверка акционного товара {ProductName} по акции '{PromotionName}'. Скидка: {PercentDiscount}%.");
    }

    public int CalculateLoyaltyPoints()
    {
        return (int)(CalculateTotal() * 0.05M) + LoyaltyPointsAwarded;
    }

    public void ExtendPromo(int days)
    {
        ExpirationDate = ExpirationDate.AddDays(days);
        Console.WriteLine($"Срок акции '{PromotionName}' продлен на {days} дней. Новая дата: {ExpirationDate:dd.MM.yyyy}.");
    }

    public void SetTargetAudience(string audience)
    {
        TargetAudience = audience;
    }

    public void AddCustomerCategory(string category)
    {
        EligibleCustomerCategories.Add(category);
    }

    public void RecordPromoUsage(string client)
    {
        PromoHistory.Add($"[{DateTime.Now:dd.MM.yyyy}] Клиент: {client}, промокод: {PromoCode}");
    }

    public decimal CalculateCashback()
    {
        return CalculateTotal() * (CashbackPercent / 100M);
    }

    public bool IsEligible(string customerCategory)
    {
        return EligibleCustomerCategories.Contains(customerCategory);
    }
}

public class FreeLine : OrderLine, ITrackable
{
    public decimal Prepayment { get; set; }
    public string SponsorName { get; set; }
    public string TrackingNumber { get; set; }
    public string GrantNumber { get; set; }
    public bool RequiresDocumentation { get; set; }

    public string RecipientOrganization { get; set; }
    public string TaxExemptionCertificate { get; set; }
    public bool IsDeliveredToBeneficiary { get; set; }
    public DateTime GrantExpiryDate { get; set; }

    public List<string> BeneficiaryRegistry { get; private set; } = new List<string>();
    public HashSet<string> ApprovedDonationPurposes { get; private set; } = new HashSet<string>();
    public string VolunteerCoordinator { get; set; }
    public string SupervisoryAuthority { get; set; }

    public FreeLine(string productName, decimal price, decimal prepayment)
        : base(productName, price)
    {
        Prepayment = prepayment;
        SponsorName = "Аноним";
        TrackingNumber = "Н/Д";
        GrantNumber = "БЕЗ_ГРАНТА";
        RequiresDocumentation = false;
        RecipientOrganization = "Благотворительный фонд";
        TaxExemptionCertificate = "№-НАЛОГ-2026";
        IsDeliveredToBeneficiary = false;
        GrantExpiryDate = DateTime.Now.AddYears(1);
        VolunteerCoordinator = "Не назначен";
        SupervisoryAuthority = "Минсоцразвития";
    }

    public override decimal CalculateTotal()
    {
        decimal total = Price * 1.25M * 1.01M + 10 - Prepayment;
        return total < 0 ? 0 : total;
    }

    public override decimal CalculateTotal(decimal subsidy)
    {
        decimal total = Price * 1.25M * 1.01M + 10 - Prepayment - subsidy;
        return total < 0 ? 0 : total;
    }

    public void GetSponsorInfo()
    {
        Console.WriteLine($"Спонсор бесплатного товара: {SponsorName}");
    }

    public override void ApplyQualityCheck()
    {
        Console.WriteLine($"Проверка благотворительного груза {ProductName} от спонсора {SponsorName}. Грант: {GrantNumber}.");
    }

    public void ConfirmDeliveryToBeneficiary()
    {
        IsDeliveredToBeneficiary = true;
        Console.WriteLine($"Груз {ProductName} успешно вручен получателю: {RecipientOrganization}.");
    }

    public void PrintTaxCertificate()
    {
        Console.WriteLine($"Налоговый сертификат для груза {ProductName}: {TaxExemptionCertificate}.");
    }

    public bool ValidateGrant()
    {
        return DateTime.Now <= GrantExpiryDate && GrantNumber != "БЕЗ_ГРАНТА";
    }

    public void AddBeneficiary(string person)
    {
        BeneficiaryRegistry.Add(person);
    }

    public void AddPurpose(string purpose)
    {
        ApprovedDonationPurposes.Add(purpose);
    }

    public void AssignCoordinator(string coordinator)
    {
        VolunteerCoordinator = coordinator;
        Console.WriteLine($"Координатором груза {ProductName} назначен: {coordinator}");
    }

    public void GenerateAuditSummary()
    {
        Console.WriteLine($"Аудиторская справка: {ProductName}, организация: {RecipientOrganization}, получателей: {BeneficiaryRegistry.Count}.");
    }

    void ITrackable.UpdateTracking(string status)
    {
        Console.WriteLine($"[Явная реализация ITrackable] Отслеживание ({TrackingNumber}): {status}");
    }

    void ITrackable.UpdateTracking(string status, string location)
    {
        Console.WriteLine($"[Явная реализация ITrackable] Отслеживание ({TrackingNumber}): {status}, локация: {location}");
    }
}

public class Order<T> where T : OrderLine
{
    private List<T> lines = new List<T>();
    private Dictionary<Guid, T> lineDictionary = new Dictionary<Guid, T>();
    private HashSet<string> uniqueCategories = new HashSet<string>();

    public string OrderNumber { get; set; }
    public decimal TotalOrderCost { get; private set; }

    public event Action<string> OrderNotification;

    public Order(string orderNumber)
    {
        OrderNumber = orderNumber;
        TotalOrderCost = 0;
    }

    public void AddLine(T line)
    {
        lines.Add(line);
        lineDictionary[line.ProductId] = line;
        uniqueCategories.Add(line.Category);
        TotalOrderCost += line.CalculateTotal() + line.GetShippingCost();
        OrderNotification?.Invoke($"В заказ {OrderNumber} добавлена позиция: {line.ProductName}");
    }

    public void AddLine(T line, bool isExpress)
    {
        line.IsExpressDelivery = isExpress;
        AddLine(line);
    }

    public T FindLineById(Guid id)
    {
        lineDictionary.TryGetValue(id, out T found);
        return found;
    }

    public List<T> FilterLines(Func<T, bool> predicate)
    {
        return lines.Where(predicate).ToList();
    }

    public void PrintReceipt()
    {
        Console.WriteLine($"=== ЧЕК ЗАКАЗА {OrderNumber} ===");
        Console.WriteLine($"Уникальные категории в заказе: {string.Join(", ", uniqueCategories)}");
        foreach (var line in lines)
        {
            line.PrintWarning();
            Console.WriteLine(line.GetProductDetails());
            Console.WriteLine($"Стоимость строки (с доставкой): {line.CalculateTotal() + line.GetShippingCost()}\n");
        }
        Console.WriteLine($"ИТОГО К ОПЛАТЕ: {TotalOrderCost}");
        Console.WriteLine("================================");
    }
}

public class BatchProcessor<T> where T : OrderLine
{
    private List<T> batchItems = new List<T>();

    public void AddItem(T item)
    {
        batchItems.Add(item);
    }

    public void ProcessAll()
    {
        Console.WriteLine("=== ПАКЕТНАЯ ПРОВЕРКА ПОЗИЦИЙ ===");
        foreach (var item in batchItems)
        {
            item.ApplyQualityCheck();
        }
        Console.WriteLine("================================");
    }
}

public class TrackingService
{
    private readonly ITrackable _trackable;

    public TrackingService(ITrackable trackable)
    {
        _trackable = trackable;
    }

    public void LogStatus(string status)
    {
        _trackable.UpdateTracking(status);
    }

    public void LogStatus(string status, string location)
    {
        _trackable.UpdateTracking(status, location);
    }
}

public class ReturnService
{
    private readonly IReturnable _returnable;

    public ReturnService(IReturnable returnable)
    {
        _returnable = returnable;
    }

    public bool ExecuteReturn(string reason)
    {
        return _returnable.RequestReturn(reason);
    }
}

public class OrderProcessingService
{
    private readonly IOrderLine _orderLine;

    public OrderProcessingService(IOrderLine orderLine)
    {
        _orderLine = orderLine;
    }

    public void ProcessLine()
    {
        Console.WriteLine($"Сервис обработки позиции: {_orderLine.ProductName}");
        _orderLine.ApplyQualityCheck();
        Console.WriteLine($"Итоговая стоимость с доставкой: {_orderLine.CalculateTotal() + _orderLine.GetShippingCost()} руб.");
    }
}

StandardLine standardItem = new StandardLine("Катушка медной проволоки", 1000, 3);
standardItem.Weight = 2.5M;
standardItem.OriginCountry = "Россия";
standardItem.PriorityLevel = 1;
standardItem.Category = "Промышленность";
standardItem.AddTag("Металл");
standardItem.AddTag("Электротехника");
standardItem.AddInternalNote("Проверена изоляция бухты");
standardItem.RegisterSerialNumber("SN-CU-001");
standardItem.RegisterSerialNumber("SN-CU-002");
standardItem.AddCertificate("ГОСТ-Р-54321");
standardItem.Palletize();
standardItem.MarkAsInspected();

SpecialLine specialItem = new SpecialLine("Бочки с метиламином", 500, 20);
specialItem.Category = "Химия";
specialItem.PromoCode = "ХИМИЯ2026";
specialItem.UpdatePrice(600);
specialItem.UpdatePrice(600, 1.1M);
specialItem.IsFragile = true;
specialItem.OriginCountry = "Германия";
specialItem.PromotionName = "Оптовая скидка";
specialItem.AddTag("Химия");
specialItem.AddTag("ОпасныйГруз");
specialItem.AddCustomerCategory("Оптовики");
specialItem.AddCustomerCategory("Лаборатории");
specialItem.RecordPromoUsage("ООО ХимПром");
specialItem.ApplyPromo("ХИМИЯ2026", 10);
specialItem.ExtendPromo(14);

FreeLine freeItem = new FreeLine("Клетка с обезьянами", 150, 120);
freeItem.Category = "Зоология";
freeItem.TrackingNumber = "TRK987654321";
freeItem.SponsorName = "Фонд защиты животных";
freeItem.GrantNumber = "ГРАНТ-2026-ЭКО";
freeItem.OriginCountry = "Бразилия";
freeItem.AddTag("Животные");
freeItem.AddTag("Благотворительность");
freeItem.AddBeneficiary("Зоопарк г. Тюмени");
freeItem.AddPurpose("Сохранение редких видов");
freeItem.AssignCoordinator("Иванов И.И.");
freeItem.PrintTaxCertificate();
freeItem.ConfirmDeliveryToBeneficiary();
freeItem.GenerateAuditSummary();

BulkyStandardLine bulkyItem = new BulkyStandardLine("Промышленный станок", 50000, 1, 1500);
bulkyItem.Category = "Оборудование";
bulkyItem.Weight = 500M;
bulkyItem.TrackingNumber = "BLK123456789";
bulkyItem.RequiredLoaders = 4;
bulkyItem.TransportType = "Грузовой автопоезд";
bulkyItem.OriginCountry = "Беларусь";
bulkyItem.AddTag("Тяжеловес");
bulkyItem.AddTag("Негабарит");
bulkyItem.AddCheckpoint("Таможенный терминал");
bulkyItem.AddCheckpoint("Складской комплекс");
bulkyItem.AddEquipment("Траверсы грузоподъемные");
bulkyItem.IssueUnloadingPermit("РАЗР-ТЮМ-2026-09");
bulkyItem.AssignCrane();
bulkyItem.UpdateRoute("Трасса М-12 через распределительный хаб");

PackageSize newBox = new PackageSize(0, 0, 0, 3, 3, 3);
standardItem.ChangeSizeBox(newBox);

standardItem.LineStatusChanged += (source, msg) =>
{
    Console.WriteLine($"[Событие строки]: {msg}");
};
standardItem.UpdateStatus("Готов к отправке");

PriceModifier holidayDiscount = (basePrice, factor) => basePrice * factor;
specialItem.ApplyCustomDiscount(0.95M, holidayDiscount);

Order<OrderLine> generalOrder = new Order<OrderLine>("№ 001-ОБЩИЙ");
generalOrder.OrderNotification += msg =>
{
    Console.WriteLine($"[Событие заказа]: {msg}");
};

generalOrder.AddLine(standardItem);
generalOrder.AddLine(specialItem);
generalOrder.AddLine(freeItem);
generalOrder.AddLine(bulkyItem, true);

generalOrder.PrintReceipt();

Dictionary<string, OrderLine> catalogByBarcode = new Dictionary<string, OrderLine>
{
    ["BAR-1001"] = standardItem,
    ["BAR-1002"] = specialItem,
    ["BAR-1003"] = freeItem,
    ["BAR-1004"] = bulkyItem
};

Console.WriteLine("=== ПОИСК В СЛОВАРЕ КАТАЛОГА ПО ШТРИХКОДУ ===");
if (catalogByBarcode.TryGetValue("BAR-1001", out OrderLine foundItem))
{
    Console.WriteLine($"По штрихкоду BAR-1001 найден товар: {foundItem.ProductName}, себестоимость: {foundItem.Price} руб.");
}
Console.WriteLine();

Console.WriteLine("=== ФИЛЬТРАЦИЯ ЛИНИЙ ЗАКАЗА ЧЕРЕЗ ЛЯМБДА-ВЫРАЖЕНИЯ ===");
var expensiveItems = generalOrder.FilterLines(line => line.CalculateTotal() > 1000M);
Console.WriteLine($"Товары дороже 1000 руб. (найдено: {expensiveItems.Count}):");
foreach (var item in expensiveItems)
{
    Console.WriteLine($"- {item.ProductName} (Категория: {item.Category}, Итого: {item.CalculateTotal()} руб.)");
}
Console.WriteLine();

var fragileItems = generalOrder.FilterLines(line => line.IsFragile);
Console.WriteLine($"Хрупкие товары (найдено: {fragileItems.Count}):");
foreach (var item in fragileItems)
{
    Console.WriteLine($"- {item.ProductName}");
}
Console.WriteLine();

Order<StandardLine> standardOrder = new Order<StandardLine>("№ 002-СТАНДАРТ");
standardOrder.AddLine(standardItem);

BatchProcessor<OrderLine> batch = new BatchProcessor<OrderLine>();
batch.AddItem(standardItem);
batch.AddItem(specialItem);
batch.AddItem(freeItem);
batch.AddItem(bulkyItem);
batch.ProcessAll();

freeItem.GetSponsorInfo();

TrackingService bulkyTrackingService = new TrackingService(bulkyItem);
bulkyTrackingService.LogStatus("Прибыл на склад");
bulkyTrackingService.LogStatus("Передан в службу доставки", "Тюмень");

ReturnService bulkyReturnService = new ReturnService(bulkyItem);
bulkyReturnService.ExecuteReturn("Не подошел по размеру");

TrackingService freeTrackingService = new TrackingService(freeItem);
freeTrackingService.LogStatus("В пути");
freeTrackingService.LogStatus("Прибыл в терминал", "Москва");

OrderProcessingService standardProcessingService = new OrderProcessingService(standardItem);
standardProcessingService.ProcessLine();

OrderProcessingService specialProcessingService = new OrderProcessingService(specialItem);
specialProcessingService.ProcessLine();

Партия Катушка медной проволоки (партия: BATCH-2026-01) упакована на паллету.
Товар Катушка медной проволоки отмечен как прошедший первичный осмотр.
Срок акции 'Оптовая скидка' продлен на 14 дней. Новая дата: 06.12.2026.
Координатором груза Клетка с обезьянами назначен: Иванов И.И.
Налоговый сертификат для груза Клетка с обезьянами: №-НАЛОГ-2026.
Груз Клетка с обезьянами успешно вручен получателю: Благотворительный фонд.
Аудиторская справка: Клетка с обезьянами, организация: Благотворительный фонд, получателей: 1.
Выдано разрешение на разгрузку Промышленный станок: РАЗР-ТЮМ-2026-09
Для погрузки крупногабаритного товара Промышленный станок назначен автокран.
Маршрут крупногабаритного груза изменен: Трасса М-12 через распределительный хаб
[Событие строки]: Статус позиции 'Катушка медной проволоки' изменен на 'Готов к отправке'.
[Событие заказа]: В заказ № 001-ОБЩИЙ добавлена позиция: Катушка медной проволоки
[Событие заказа]: В заказ № 001-ОБЩИЙ добавлена позиция: Бочки с метиламином
[Со